# ISEWM - Intelligent Scholarship Ranking — Google Colab Training

This notebook trains the scholarship ranking model (`ml/pipeline.py`'s Random Forest) on Google Colab, including **any historical training data** you upload.

**It works 100% even if your uploaded file:**
- ✅ has annual income + ITR columns
- ✅ is missing the income column (rows keep a neutral need score)
- ✅ is missing the ITR column
- ✅ has messy values (`"PHP 120,000.00"`, aliases like `family_income`, `gender`, ...)

A GPU is **not** needed — Random Forest is a CPU model. Just run the cells top to bottom (**Runtime ▸ Run all**).

> No Supabase keys needed in Colab — training uses the synthetic population + historical data you provide.

In [ ]:
# Cell 1 — Install dependencies (Colab already ships numpy/scipy/pandas)
!pip -q install scikit-learn joblib openpyxl pandas
print("Dependencies ready.")

In [ ]:
# Cell 2 — Get the ml/ source files into this notebook
#
# OPTION A (fastest): paste your GitHub repo URL below and run this cell.
# OPTION B: run the cell with REPO_URL empty — you will be asked to upload
#           either your project ZIP, or just these 5 files:
#             config.py, features.py, csv_data.py, rank_model.py, __init__.py
#           (all inside the ml/ folder of your project)

REPO_URL = ""  # e.g. "https://github.com/yourname/capstone-machine-learning.git"

import shutil
import sys
from pathlib import Path

TARGET = Path("/content/ml")

def find_ml_dir():
    """Locate a directory holding the 4 core ranking modules."""
    for cfg in Path("/content").rglob("config.py"):
        d = cfg.parent
        if all((d / f).exists() for f in ("features.py", "csv_data.py", "rank_model.py", "__init__.py")):
            return d
    return None

if REPO_URL:
    import subprocess
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/repo"], check=True)
    src = find_ml_dir()
else:
    from google.colab import files as colab_files
    print("Upload your project ZIP — or the 5 ml/ files listed above (+ your training_data.csv).")
    up = colab_files.upload()
    for name in up:
        if name.endswith(".zip"):
            subprocess = __import__("subprocess")
            subprocess.run(["unzip", "-oq", name, "-d", "/content"], check=True)
    src = find_ml_dir()
    if src is None:
        print("ZIP/clone not detected — treat uploaded files as the ml/ folder itself.")
        TARGET.mkdir(parents=True, exist_ok=True)
        for name, content in colab_files.upload().items():
            (TARGET / name).write_bytes(content)
        src = TARGET

assert src is not None, "Could not find ml/ files — upload the project ZIP or the 5 listed files."
if src.resolve() != TARGET.resolve():
    shutil.copytree(src, TARGET, dirs_exist_ok=True)

sys.path.insert(0, "/content")
from ml import config, csv_data, features, rank_model
print(f"ml/ ready at {TARGET} | stack: {config.stack_info()}")

In [ ]:
# Cell 3 — (Optional, but recommended) Upload YOUR historical training data.
# Set UPLOAD_DATA = False to use the bundled sample file (ml/training_data.csv) instead.

UPLOAD_DATA = True

if UPLOAD_DATA:
    from google.colab import files as colab_files
    print("Choose your historical file: training_data.csv / training_data.xlsx (income column optional).")
    up = colab_files.upload()
    for name, content in up.items():
        suffix = ".xlsx" if name.lower().endswith(".xlsx") else ".csv"
        (TARGET / f"training_data{suffix}").write_bytes(content)
        print(f"Saved as ml/training_data{suffix}")

rows, warnings = csv_data.load_training_csv()
print(f"\nHistorical rows usable for training: {len(rows)}")
for w in warnings:
    print("  WARN:", w)
if rows:
    import json
    print("\nSample parsed row:")
    print(json.dumps(rows[0], indent=2, default=str))

In [ ]:
# Cell 4 — Train the Random Forest (same configuration the production pipeline uses)

res = rank_model.train(
    [],                      # real applicants come from the app's OCR pipeline; here we train on synthetic + historical
    seed=2026,
    n_trees=100,
    n_synth=800,
    max_depth=8,
)

print("=== TRAINING RESULT ===")
print(f"Total fit rows      : {res['n_samples']}  (synthetic {res['n_synthetic']} + historical {res['n_csv']} + real {res['n_real']})")
print(f"Accuracy            : {res['accuracy']:.4f}")
print(f"Precision / Recall  : {res['precision']:.4f} / {res['recall']:.4f}")
print(f"F1                  : {res['f1']:.4f}")
print(f"ROC-AUC             : {res['roc_auc']}")
print(f"R2 (priority score) : {res['r2']:.4f}")
print(f"RMSE                : {res['rmse']:.4f}")
print(f"Confusion matrix    : {res['confusion_matrix']}")
print(f"Trained in          : {res['train_ms']} ms")
print(f"Model saved         : {res['model_path']}")
print("\nDataset analytics:", res["dataset"].get("sources"), "|", res["dataset"].get("income"))
print("Feature importance:")
for k, v in sorted(res["feature_importance"].items(), key=lambda kv: -kv[1]):
    print(f"  {k:24s} {v}")
for w in res.get("csv_warnings") or []:
    print("csv warning:", w)
print("\nNOTE: accuracy/R2 look artificially perfect because the ranking target is a deterministic"
      " function of income (W_NEED=1.0, W_OCR=0.0, W_ACADEMIC=0.0 in ml/config.py) — the forest simply"
      " learns need = income. That matches the capstone's documented design: income is the ranking driver"
      " and OCR alignment acts as a trust gate, not a score.")

In [ ]:
# Cell 5 — Demo ranking (same logic as the web app: ITR-verified first, then most in need)

sample_applicants = [
    {"application_id": 1, "student_name": "Ana Reyes",     "student_number": "2023-0011", "program": "BSIT 3-1", "year_level": "3", "sex": "Female", "declared_income": 58000.0},
    {"application_id": 2, "student_name": "Carlo Magno",   "student_number": "2022-0044", "program": "BSCS 4-1", "year_level": "4", "sex": "Male",   "declared_income": 520000.0},
    {"application_id": 3, "student_name": "Maria Santos",  "student_number": "2024-0107", "program": "BSIT 1-1", "year_level": "1", "sex": "Female", "declared_income": 145000.0},
    {"application_id": 4, "student_name": "Juan Dela Cruz","student_number": "2022-0033", "program": "BSBA 3-2", "year_level": "3", "sex": "Male",   "declared_income": 88000.0},
]
ocr_results = [
    {"ocr_value": 58000.0,  "match": {"align_pct": 99.2, "status": "match"}, "low_confidence": False},
    {"ocr_value": 525000.0, "match": {"align_pct": 98.5, "status": "match"}, "low_confidence": False},
    {"ocr_value": 143000.0, "match": {"align_pct": 95.1, "status": "near"},  "low_confidence": False},
    {"ocr_value": 91000.0,  "match": {"align_pct": 88.0, "status": "mismatch"}, "low_confidence": False},
]

ranked = rank_model.predict([features.build_row(a, o) for a, o in zip(sample_applicants, ocr_results)], res)

import pandas as pd
cols = ["rank", "student_name", "income_source", "income_used", "financial_need_score",
        "itr_trust", "priority_score", "prediction", "probability"]
display(pd.DataFrame(ranked)[cols])

In [ ]:
# Cell 6 — Download the results

from google.colab import files as colab_files

# 1. The trained model artifact
colab_files.download(res["model_path"])

# 2. The demo ranking as CSV
import csv as _csv
with open("colab_ranking_demo.csv", "w", newline="", encoding="utf-8") as f:
    writer = _csv.DictWriter(f, fieldnames=list(ranked[0].keys()))
    writer.writeheader()
    writer.writerows(ranked)
colab_files.download("colab_ranking_demo.csv")

print("Model file  : ", res["model_path"].split("/")[-1], f"({res['model_size_kb']} KB) — put it in ml/models/ on your machine to reuse it.")
print("Demo ranking: colab_ranking_demo.csv")

## How to reuse the Colab results locally
1. Download `rf_ranking_seed2026.joblib` (Cell 6).
2. On your machine, put it at **`ml/models/rf_ranking_seed2026.joblib`** (the path your seed/config implies).
3. Note: the production pipeline **re-trains and overwrites** this file on every ranking run, so the local app always trains fresh — the Colab artifact is useful as evidence/analysis in your capstone (screenshots of metrics, feature importance, ranking demo).